# 02 - Data prep

Objetivo: preparar os dados de forma reprodutivel e sem vazamento. Limpezas deterministicas entram aqui; imputacao, escala e encoding ficam dentro do pipeline.


In [ ]:
# Load libraries and data
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
TARGET = 'Churn'
ID_COLS = ['customerID']
df_raw = pd.read_csv('../data/raw/telco_churn_raw.csv')


In [ ]:
# Clean text fields and convert types
df = df_raw.copy()
df.columns = df.columns.str.strip()
text_cols = df.select_dtypes(include=['object', 'string']).columns
for col in text_cols:
    df[col] = df[col].astype('string').str.strip()

df[TARGET] = df[TARGET].map({'Yes': 1, 'No': 0}).astype(int)
df['SeniorCitizen'] = df['SeniorCitizen'].map({1: 'Yes', 0: 'No'}).astype('string')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace('', np.nan), errors='coerce')
df['tenure'] = pd.to_numeric(df['tenure'], errors='coerce')
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')
df.head()


In [ ]:
# Validate basic consistency rules
checks = pd.Series({
    'target_binario': set(df[TARGET].unique()).issubset({0, 1}),
    'sem_id_duplicado': not df['customerID'].duplicated().any(),
    'tenure_nao_negativo': (df['tenure'].dropna() >= 0).all(),
    'monthly_charges_positivo': (df['MonthlyCharges'].dropna() > 0).all(),
    'total_charges_missing_apenas_tenure_zero': (df.loc[df['TotalCharges'].isna(), 'tenure'] == 0).all()
})
display(checks.to_frame('ok'))
assert checks.all()


In [ ]:
# Split before fitting any statistical transformation
X = df.drop(columns=ID_COLS + [TARGET])
y = df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

X_train.shape, X_val.shape, X_test.shape


In [ ]:
# Build preprocessing pipeline
numeric_features = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['number', 'bool']).columns.tolist()

try:
    one_hot = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    one_hot = OneHotEncoder(handle_unknown='ignore', sparse=False)

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', one_hot)
    ]), categorical_features),
])

numeric_features, categorical_features
